In [ ]:
# Import
import os
import cv2
import shutil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# 目标提取

In [ ]:
src_path = os.path.join(os.getcwd(),"Data")
dst_path = os.path.join(os.getcwd(),"Processed_Data")
# 先要将图片中的目标提取出来
def process_data(src,dst,type):
    src_path = os.path.join(src,type)
    dst_path = os.path.join(dst,type)
    os.makedirs(dst_path,exist_ok=True)
    tar_dic = {}
    label_path = os.path.join(src_path,"labels")
    try:
        for f_name in os.listdir(os.path.join(label_path)):
            name,number,_ = f_name.split("-")
            with open(os.path.join(label_path,f_name),"r") as f:
                lines = f.readlines()
                for line in lines:
                    _,x_center,y_center,width,height = line.strip().split(" ")
                    x_center = float(x_center)
                    y_center = float(y_center)
                    width = float(width)
                    height = float(height)
            tar_dic[(name,number)] = [x_center,y_center,width,height]
        image_path = os.path.join(src_path,"images")
        for f_name in os.listdir(image_path):
            name,number,_ = f_name.split("-")
            if (name,number) in tar_dic:
                img = cv2.imread(os.path.join(image_path,f_name))
                h,w,_ = img.shape
                x_center,y_center,width,height = tar_dic[(name,number)]
                x_center = int(x_center*w)
                y_center = int(y_center*h)
                width = int(width*w)
                height = int(height*h)
                x1 = max(0,x_center - width//2)
                y1 = max(0,y_center - height//2)
                x2 = min(w,x_center + width//2)
                y2 = min(h,y_center + height//2)
                cropped_img = img[y1:y2,x1:x2]
                cv2.imwrite(os.path.join(dst_path,f"{name}_{number}.jpg"),cropped_img)
            else:
                print(f"Warning: No label for image {f_name}")
    except Exception as e:
        print(f"{type} error: {name}_{number} - {str(e)}")

process_data(src_path,dst_path,"train")
process_data(src_path,dst_path,"valid")
process_data(src_path,dst_path,"test")
    

# Neuro Network (CNN) 

In [ ]:
# path = os.path.join(os.getcwd(),"Data")

# def creat_NN_data(path,mode="train"):
#     save_path = os.path.join(path,f"NN_{mode}")
#     os.makedirs(save_path,exist_ok=True)
#     img_path = os.path.join(path,mode+"/images")

#     for f_name in os.listdir(img_path):
#         if f_name.lower().endswith(".jpg"):
#             label,count,_ = f_name.split("-")
#             dst_path = os.path.join(save_path,label)
#             os.makedirs(dst_path,exist_ok=True)
#             src_path = os.path.join(img_path,f_name)
#             dst_path = os.path.join(dst_path,f"{count}.jpg")
#             shutil.copy2(src_path,dst_path)
             
# creat_NN_data(path,"train")
# creat_NN_data(path,"test")
# creat_NN_data(path,"valid")
        
    

In [ ]:
# NN_train_path = os.path.join(path,f"NN_train")
# NN_test_path = os.path.join(path,f"NN_test")
# NN_val_path = os.path.join(path,f"NN_valid")

# transform = transforms.Compose([
#     # 统一将数据先搞成640*640的（其实有点太大了）
#     transforms.Resize((640, 640)),
#     # 将PIL转成pytorch的tensor
#     transforms.ToTensor(),
#     # 按照ImageNet上面统计过的均值和标准差进行归一化（提升训练效果）
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# train_dataset = datasets.ImageFolder(NN_train_path, transform=transform)
# val_dataset = datasets.ImageFolder(NN_val_path, transform=transform)

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_classes = len(train_dataset.classes)

# model = models.resnet18(pretrained=True)
# model.fc = nn.Linear(model.fc.in_features, num_classes)
# model = model.to(device)

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# num_epochs = 10
# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()

#     model.eval()
#     correct = total = 0
#     with torch.no_grad():
#         for images, labels in val_loader:
#             images, labels = images.to(device), labels.to(device)
#             outputs = model(images)
#             _, predicted = torch.max(outputs, 1)
#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()

#     print(f'Epoch {epoch+1}/{num_epochs}: Loss={running_loss/len(train_loader):.4f}, Val Acc={correct/total:.4f}')

# torch.save(model.state_dict(), "crop_pest_resnet18.pth")

